In [1]:
from pathlib import Path
from typing import List
from dotenv import load_dotenv
import os
import certifi

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pypdf import PdfReader
import docx2txt
from langchain_community.vectorstores import FAISS

d:\Document\AI AGENT\ChatBotGPT\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_15432\2595409955.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:

Path("uploads").mkdir(exist_ok=True)
Path("chroma_db").mkdir(exist_ok=True)
from langchain_huggingface import HuggingFaceEmbeddings

#Embeddings Model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Loading Chroma...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5630.82it/s]


Loading Chroma...


In [3]:
def read_file_text(file_path: str) -> str:
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        reader = PdfReader(file_path)
        text = ""

        for page in reader.pages:
            text += page.extract_text() or ""
            text += "\n"

        return text

    if suffix == ".docx":
        return docx2txt.process(file_path)

    if suffix in [".txt", ".md", ".py", ".csv"]:
        return path.read_text(encoding="utf-8", errors="ignore")

    raise ValueError("Unsupported file type. Upload PDF, DOCX, TXT, MD, PY, or CSV.")

In [4]:

def add_document_to_rag(file_path: str, thread_id: str):
    print("\n========== RAG START ==========")
    print(f"File path: {file_path}")
    print(f"Thread ID: {thread_id}")

    # Step 1: Extract text
    print("\n[1/5] Reading document...")

    text = read_file_text(file_path)
    print(text)

    print(f"Text extracted successfully.")
    print(f"Characters extracted: {len(text)}")

    if not text.strip():
        raise ValueError(
            "No text could be extracted from this file."
        )

    # Step 2: Split text
    print("\n[2/5] Splitting document...")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=150
    )

    chunks = splitter.split_text(text)
    print(chunks)

    print(f"Chunks created: {len(chunks)}")

    if not chunks:
        raise ValueError("No chunks were created.")

    # Step 3: Create Documents
    print("\n[3/5] Creating LangChain documents...")

    docs: List[Document] = [
        Document(
            page_content=chunk,
            metadata={
                "thread_id": thread_id,
                "source": Path(file_path).name
            }
        )
        for chunk in chunks
    ]
    
    print(docs)

    print(f"Documents created: {len(docs)}")

    

    print("\n[5/5] Adding documents to Chroma...")
    try:
        vector_store = FAISS.from_documents(docs, embeddings)
        vector_store.save_local("chroma_db")
    except Exception as e:
        import traceback
        print("❌ FAILED in add_documents/persist step:")
        traceback.print_exc()
        raise

    print("Documents successfully added to Chroma.")

    print("\n========== RAG COMPLETE ==========\n")

    return {
        "filename": Path(file_path).name,
        "chunks": len(docs)
    }


In [5]:
add_document_to_rag("D:/Document/AI AGENT/ChatBotGPT/uploads/2e91fa48-899e-47a1-887c-c1acdc88584b_Anuj_Kumar_Sao_Electrician_Resume_Updated.pdf","user_80")


========== RAG START ==========
File path: D:/Document/AI AGENT/ChatBotGPT/uploads/2e91fa48-899e-47a1-887c-c1acdc88584b_Anuj_Kumar_Sao_Electrician_Resume_Updated.pdf
Thread ID: user_80

[1/5] Reading document...
Anuj Kumar Sao — Electrician Resume
Page 1
 ANUJ KUMAR SAO
ELECTRICIAN | STEEL PLANT & INDUSTRIAL ELECTRICAL MAINTENANCE
Mobile: 97132 78484 | Madan Bigha, Chakand, Gaya, Bihar
 CAREER OBJECTIVE
Dedicated and safety-conscious Electrician with extensive hands-on experience in steel plants and industrial environments. Seeking a
responsible position where I can apply my practical knowledge of electrical maintenance, equipment troubleshooting, motors, panels,
wiring and plant electrical systems, while contributing to safe, reliable and efficient plant operations.
PROFESSIONAL SUMMARY
 Extensive practical experience as an electrician in steel and industrial plants.
 Experienced in electrical maintenance, fault finding, wiring, motors, panels and routine plant support.
 Able to w

{'filename': '2e91fa48-899e-47a1-887c-c1acdc88584b_Anuj_Kumar_Sao_Electrician_Resume_Updated.pdf',
 'chunks': 4}

In [6]:
def retrieve_from_rag(query: str, thread_id:str, k: int = 4)-> str:
    DB_PATH = "chroma_db"
    vector_store = FAISS.load_local(
            folder_path=DB_PATH,
            embeddings=embeddings,
            allow_dangerous_deserialization=True
        )
    docs = vector_store.similarity_search(
        query,
        k=k,
        filter={"thread_id":thread_id}
    )  
    
    if not docs:
        return "No relevant uploaded document content found."
    
    results = []
    
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source","uploaded document")
        results.append(
            f"[Source {i}: {source}]\n{doc.page_content}"
        )
    
    print(results)    
    return "\n\n".join(results) 

In [7]:
retrieve_from_rag("What is mentioned skill in uloaded document","user_80")

['[Source 1: 2e91fa48-899e-47a1-887c-c1acdc88584b_Anuj_Kumar_Sao_Electrician_Resume_Updated.pdf]\nAnuj Kumar Sao — Electrician Resume\nPage 2\n EXPERIENCE CERTIFICATE\n Shri Ram Hi-Tech Steel & Power Pvt. Ltd.', "[Source 2: 2e91fa48-899e-47a1-887c-c1acdc88584b_Anuj_Kumar_Sao_Electrician_Resume_Updated.pdf]\nH.R.G.\nKarnataka\n2006 – 2008\nHelper\nGorbal Power House\nGoa\nEDUCATION\nQualification\nBoard\nResult\nIntermediate\nBihar School Examination Board (BSEB)\nFirst Division\nMatriculation\nBihar School Examination Board (BSEB)\nFirst Division\nKEY SKILLS\nIndustrial Electrical Maintenance \x7f Steel Plant Electrical Systems \x7f Electrical Wiring & Installation \x7f Motor & Panel Maintenance \x7f\nElectrical Fault Finding & Troubleshooting \x7f Preventive Maintenance \x7f Safety Procedures & PPE Compliance \x7f Teamwork & Shift Work\nPERSONAL DETAILS\nFather's Name\nShiv Kumar Sao\nDate of Birth\n25/02/1992\nAddress\nMadan Bigha, Chakand, Gaya, Bihar\nMobile\n97132 78484\nNationali

"[Source 1: 2e91fa48-899e-47a1-887c-c1acdc88584b_Anuj_Kumar_Sao_Electrician_Resume_Updated.pdf]\nAnuj Kumar Sao — Electrician Resume\nPage 2\n EXPERIENCE CERTIFICATE\n Shri Ram Hi-Tech Steel & Power Pvt. Ltd.\n\n[Source 2: 2e91fa48-899e-47a1-887c-c1acdc88584b_Anuj_Kumar_Sao_Electrician_Resume_Updated.pdf]\nH.R.G.\nKarnataka\n2006 – 2008\nHelper\nGorbal Power House\nGoa\nEDUCATION\nQualification\nBoard\nResult\nIntermediate\nBihar School Examination Board (BSEB)\nFirst Division\nMatriculation\nBihar School Examination Board (BSEB)\nFirst Division\nKEY SKILLS\nIndustrial Electrical Maintenance \x7f Steel Plant Electrical Systems \x7f Electrical Wiring & Installation \x7f Motor & Panel Maintenance \x7f\nElectrical Fault Finding & Troubleshooting \x7f Preventive Maintenance \x7f Safety Procedures & PPE Compliance \x7f Teamwork & Shift Work\nPERSONAL DETAILS\nFather's Name\nShiv Kumar Sao\nDate of Birth\n25/02/1992\nAddress\nMadan Bigha, Chakand, Gaya, Bihar\nMobile\n97132 78484\nNationalit